# 04 - Full Pipeline Demo and Gradio App

Runs the complete system: **question → domain gate → retriever → LoRA LLM → safety filter → answer**, then launches the Gradio app with a public link (use it for your demo video, or as the fallback if a free Hugging Face Space is too slow).

**Needs on Drive** (produced by notebooks 01-03): `data/processed/corpus.jsonl`, `results/gate/best`, `results/retriever/best_dense`, and a LoRA adapter, e.g. `results/runs/e4_lora_rag_climate/adapter`.

⚠️ `use_context` / `use_climate` in `app/config.json` **must match how the adapter was trained** (E3 = context only; E4 = context + climate). A mismatch silently degrades answers.

## 0. Setup

In [ ]:
!nvidia-smi -L
!pip install -q gradio "transformers>=4.46" "peft>=0.13" "accelerate>=1.0" sentence-transformers faiss-cpu scikit-learn sentencepiece protobuf
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json
PROJECT = '/content/drive/MyDrive/crop-advisory-assistant'   # <- change if needed
os.chdir(PROJECT); sys.path.insert(0, f'{PROJECT}/src'); sys.path.insert(0, f'{PROJECT}/app')

In [ ]:
!python tests/test_pipeline.py | tail -2      # offline plumbing check with stand-ins

## 1. Configure
Edit the adapter path and the two prompt flags to match your best run.

In [ ]:
from pipeline import AdvisoryPipeline, PipelineConfig
cfg = PipelineConfig.from_json('app/config.json')
cfg.adapter_dir = f'{PROJECT}/results/runs/e4_lora_rag_climate/adapter'   # <- your best adapter
cfg.use_context, cfg.use_climate = True, True                              # <- must match training
cfg.retriever_mode = 'hybrid'                                              # bm25 | dense | hybrid
print(cfg)
pipe = AdvisoryPipeline(cfg)
_ = pipe.gate, pipe.retriever

## 2. Try it
Shows the English answer, the gate decision, retrieved sources, safety flags and per-stage timings.

In [ ]:
from weather import climate_from_values
def show(q, **kw):
    r = pipe.answer(q, **kw)
    print('STATUS :', r['status'], r.get('gate'))
    print('ANSWER :', r['answer'])
    if r['flags']: print('FLAGS  :', r['flags'])
    for p in r['passages']: print('  source:', p['chunk_id'])
    print('TIMINGS:', r['timings'], '\n')
    return r

Q = 'Should I be worried about late blight in my potato field this week?'
_ = show(Q, crop='potato', climate=climate_from_values(16, 92, 70))   # cool + wet
_ = show(Q, crop='potato', climate=climate_from_values(30, 40, 0))    # hot + dry

In [ ]:
_ = show('What causes rust on bean leaves and how can I prevent it?', crop='beans')
_ = show('Who won the football match yesterday?')                       # should be refused

In [ ]:
r = show('How does cassava mosaic disease spread?', crop='cassava', output_lang='rw')
print('English version:', r['answer_en'])

## 4. Launch the web app
Run this cell and open the `gradio.live` link. It stays alive while the cell runs (and ~72 h at most for the public link).

In [ ]:
import app as appmod
demo = appmod.build_demo(pipe)
demo.queue().launch(share=True, debug=False)

## 5. Deploy to Hugging Face Spaces (permanent link for the report)

1. Make sure your adapter and artifacts are final, then build the Space folder:
   ```
   !python deploy/prepare_space.py --space_dir space --adapter results/runs/e4_lora_rag_climate/adapter --llm_model Qwen/Qwen2.5-1.5B-Instruct
   ```
   (add `--no_climate` / `--no_context` if your adapter was trained without them).
2. Create a Space on huggingface.co (SDK: **Gradio**), then upload: `huggingface-cli upload YOUR_USER/crop-advisory-assistant space . --repo-type=space`.
3. **Hardware:** a 1.5B model on the free CPU tier is slow (can take minutes per answer) and tight on RAM together with the translation model. Options: use a GPU Space; or fine-tune the 0.5B model for the deployed version; or keep the Space for the link and demo from Colab. Whatever you choose, state it in the report's Deployment and Limitations sections.
4. Put the Space URL in your README and report.

In [ ]:
# !python deploy/prepare_space.py --space_dir space --adapter results/runs/e4_lora_rag_climate/adapter --llm_model Qwen/Qwen2.5-1.5B-Instruct